# Evaluate HistoOmniST and external prediction bundles

The manuscript benchmark uses fixed held-out slides and a common target
gene set. This notebook documents the data contract and invokes the
evaluator as a Python API. Third-party repositories and their checkpoints
remain governed by the original projects and are not redistributed here.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the HistoOmniST repository.")


ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(ROOT / "src"))

from histoomnist.eval.benchmark_predictions import align_genes, evaluate_prediction_bundle
from histoomnist.eval.evaluate_histoomnist_benchmark import evaluate_histoomnist_benchmark
from histoomnist.utils.config import load_config


## 1. Fixed methods, slides and genes


In [ ]:
methods = [
    "HistoOmniST", "HiST", "Hist2ST", "HisToGene", "iStar",
    "mclSTExp", "Path2Space", "sCellST", "STimage", "ST-Net", "THItoGene",
]
split_manifest = pd.read_csv(ROOT / "data/HEST-1k/splits/leave_slide_out.csv")
target_genes = (
    ROOT / "data/HEST-1k/manifests/highconf_symbol_coverage95_genes.txt"
).read_text(encoding="utf-8").splitlines()

print("Methods:", ", ".join(methods))
print(f"Common HistoOmniST target: {len(target_genes):,} genes")
display(split_manifest["split"].value_counts().rename_axis("split").to_frame("slides"))


## 2. Prediction-bundle contract

An external method writes one two-dimensional array per test slide and a
single ordered gene file. Rows must follow the spot order in the HEST
manifest. The evaluator intersects genes explicitly and rejects spot or
gene dimension mismatches.

```text
<prediction_root>/
  genes.txt
  <slide_id>.npy
  ...
```


In [ ]:
example_prediction_genes = ["EPCAM", "MKI67", "CD3D"]
example_target_genes = ["CD3D", "EPCAM", "COL1A1"]
common, target_index, prediction_index = align_genes(
    example_target_genes,
    example_prediction_genes,
)
display(
    pd.DataFrame(
        {
            "common_gene": common,
            "target_column": target_index,
            "prediction_column": prediction_index,
        }
    )
)


## 3. Evaluate one external method through the shared Python API


In [ ]:
METHOD_NAME = "Path2Space"
PREDICTION_ROOT = ROOT / "results/external_predictions/path2space"
PREDICTION_GENES = PREDICTION_ROOT / "genes.txt"

if PREDICTION_ROOT.exists() and PREDICTION_GENES.exists():
    expression_config = load_config(
        ROOT / "configs/hest1k_human_visium_expression_highconf_symbol95.yaml"
    )
    summary = evaluate_prediction_bundle(
        expression_config=expression_config,
        prediction_root=PREDICTION_ROOT,
        method_name=METHOD_NAME,
        prediction_kind="count",
        prediction_genes_path=PREDICTION_GENES,
        out_dir=ROOT / f"outputs/benchmark/{METHOD_NAME.lower()}",
        splits=["test"],
    )
    display(pd.Series(summary, name="value").to_frame())
else:
    print("Place a standardized prediction bundle at", PREDICTION_ROOT)
    print("See docs/benchmark.md and THIRD_PARTY_NOTICES.md for method-specific provenance.")


## 4. Evaluate HistoOmniST under the same held-out-slide contract


In [ ]:
RUN_HISTOOMNIST_BENCHMARK = False

if RUN_HISTOOMNIST_BENCHMARK:
    expression_config = load_config(
        ROOT / "configs/hest1k_human_visium_expression_highconf_symbol95.yaml"
    )
    sf_config = load_config(
        ROOT / "configs/hest1k_human_visium_sf_context_distribution_light.yaml"
    )
    summary = evaluate_histoomnist_benchmark(
        expression_config=expression_config,
        sf_config=sf_config,
        expression_checkpoint=ROOT / "checkpoints/hest1k_human_visium_expression/highconf_symbol95_rate/best.pt",
        sf_checkpoint=ROOT / "checkpoints/hest1k_human_visium_sf/context_distribution_light_hipt256_leave_slide_out/best.pt",
        out_dir=ROOT / "outputs/benchmark/histoomnist",
        splits=["test"],
    )
    display(pd.Series(summary, name="value").to_frame())
else:
    print("Set RUN_HISTOOMNIST_BENCHMARK=True after preparing the held-out HEST arrays.")


Formal reporting must retain the complete test split, the common gene
intersection and each method's source-faithful status. Smoke runs,
thumbnail-only pilots and reduced-gene runs are engineering checks and
must not be mixed with the manuscript benchmark.
